<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>Clinical Decisions at the Threshold</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/clinical-decisions.jpg" alt="A clinician in a white coat with a stethoscope talks with a smiling patient in glasses and a headscarf during a consultation." width="640" />
  <p><em>The cutoff is not abstract. It is a prescription in the room.</em></p>
</div>

---

## Start here
This notebook answers three questions:
1. Does the equation change move anyone's metformin prescribing tier?
2. Which direction does each patient move, and who moves?
3. What review rule should a tier change trigger?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

## Cold start
Same cohort, same split, whichever notebook you opened first. Earlier
notebooks do not need to be open.

In [ ]:
from dataclasses import replace

from data_supply_chain.bootstrap import cold_start
from data_supply_chain.egfr import add_egfr_columns

S = cold_start()
S = replace(S, df=add_egfr_columns(S.df))
print(f"{len(S.df):,} adults · source={S.source} · real={S.is_real}")

### **Metformin tiers under each equation**
Step 5 asked how well a mortality model **ranks** patients by risk, so it was
graded with a ranking metric, AUROC. This step asks a different question, so it
needs a different measurement: **does a changed eGFR push a patient across a
treatment line?** The answer is a count of patients, not a model score.

This cell does **not** train a model. It applies the FDA metformin cutoffs to
each patient's two eGFR values:

| Tier | eGFR |
|---|---|
| Standard | 45 and above |
| Caution | 30 to 44 |
| Contraindicated | below 30 |

A patient whose tier differs between the two equations could receive a
different prescribing decision. We start with one direction: patients the
race-based 2009 equation placed in a **more permissive** tier than the
race-free 2021 equation. We call this direction **masked**.

In [ ]:
from data_supply_chain.decisions import (
    decision_gate,
    masking_directions,
    medication_masking_table,
    tier_transition_table,
)

dm = S.df[S.df.diabetic].copy()
mask = medication_masking_table(dm)
directions = masking_directions(dm)

print(f"Diabetic adults: N={len(dm):,}  (Black: {int(dm.black.sum()):,})")
print("Share placed in a MORE permissive tier by the race-based equation (masked):")
for race in ("Black", "Non-Black"):
    print(f"   {race:10s} {mask['by_race_pct'].get(race, 0):5.2f}%  "
          f"(n={mask['by_race_n'].get(race, 0):,})")

print(f"\nBlack patients affected: {mask['black_masked_n']} total")
print(f"  fell below 45 (standard -> caution)        : {mask['black_cross_45']}")
print(f"  fell below 30 (caution -> contraindicated) : {mask['black_cross_30']}")

## Count both directions
The cell above shows 0.00% for Non-Black adults. That is correct, but it only
counts the masked direction. A tier can also change the other way:

- **Masked:** the race-based 2009 equation gives a more permissive tier.
- **Reverse:** the race-free 2021 equation gives a more permissive tier.

Both tables below cover the same diabetic adults counted above.

In [ ]:
print("MASKED: the race-based equation gives a more permissive tier  (tier_2009 > tier_2021)")
print(tier_transition_table(dm, "masked").to_string(index=False))
print(f"   n={sum(directions['masked_n'].values())}  ·  by race: {directions['masked_n']}")

print("\nREVERSE: the race-free equation gives a more permissive tier  (tier_2021 > tier_2009)")
print(tier_transition_table(dm, "reverse").to_string(index=False))
print(f"   n={sum(directions['reverse_n'].values())}  ·  by race: {directions['reverse_n']}")

print("\nWhich group did the equation change help?")
print("  That question is too simple. It assumes one group absorbs all of the change.")
_black_masked = int(directions["masked_n"]["Black"])
_nb_reverse = int(directions["reverse_n"]["Non-Black"])
print(f"  {_black_masked} Black patients move to a stricter tier. "
      f"{_nb_reverse} Non-Black patients move to a more permissive tier.")
print("  No patient falls on the other side of either count.")
print(f"  These are NOT netted: {_black_masked} moving to a stricter tier "
      f"(dose reduction or contraindication) is not "
      f"cancelled by {_nb_reverse} gaining eligibility.")
print("  They are different patients facing different decisions.")

In [ ]:
# Demo, take this opportunity to review the output

from IPython.display import Image, SVG, display
from pathlib import Path
import tempfile

from data_supply_chain import figures as _fig

def _show_workshop_fig(png_name, plot_name, *args):
    png = getattr(_fig, png_name, None)
    if png is not None:
        display(Image(data=png(*args), format="png"))
        return
    out = Path(tempfile.mkdtemp()) / "fig.svg"
    getattr(_fig, plot_name)(*args, out)
    display(SVG(filename=str(out)))

In [ ]:
_rates = {
    f"{key}_{direction}": directions[f"{direction}_pct"][race]
    for key, race in (("black", "Black"), ("nonblack", "Non-Black"))
    for direction in ("masked", "reverse")
}
_counts = {
    f"{key}_{direction}": directions[f"{direction}_n"][race]
    for key, race in (("black", "Black"), ("nonblack", "Non-Black"))
    for direction in ("masked", "reverse")
}

_show_workshop_fig("metformin_masking_png", "plot_metformin_masking", _rates, _counts)

### **A different question needs a different gate**
In step 5, the gate asked whether the mortality model still performed well.
Here we are auditing something different:

**Did changing the upstream equation change anyone's prescribing tier?**

For this workshop, we pre-commit to a simple review rule for the masked
direction:

**If the race-based equation placed even one Black diabetic patient in a more
permissive tier, stop and investigate.**

The review threshold is therefore **zero affected patients**.

Zero is not a clinical standard or a statistical cutoff. It is an audit choice.
We are saying that a treatment-tier change is consequential enough that we do
not want it silently absorbed into an aggregate metric.

And we do not subtract one direction from the other. A patient moving to a
more permissive tier does not cancel a different patient moving to a less
permissive one.

This gate checks only the masked direction. The reverse-direction patients
above would trip a zero threshold too, so a complete audit gates both
directions.

Crossing the threshold does not prove harm. It tells us **where review is
required**.

In [ ]:
dec_verdict, dec_why = decision_gate(mask["by_race_pct"].get("Black", 0), max_masked_pct=0.0)
print(f"Clinical decision gate: {dec_verdict}  {dec_why}")

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path
from data_supply_chain.decisions import TIER_LABELS

f = Findings.open()
f.record("act2.masking.diabetic_n", len(dm))
f.record("act2.masking.black_pct", round(mask["by_race_pct"].get("Black", 0), 2))
f.record("act2.masking.nonblack_pct", round(mask["by_race_pct"].get("Non-Black", 0), 2))
f.record("act2.masking.black_n", mask["black_masked_n"])
f.record("act2.masking.black_denominator", int(mask["by_race_n"].get("Black", 0)))
f.record("act2.masking.cross_45", mask["black_cross_45"])
f.record("act2.masking.cross_30", mask["black_cross_30"])
f.record("act2.masking.reverse_n", int(sum(directions["reverse_n"].values())))
f.record("act2.masking.reverse_nonblack_pct", round(directions["reverse_pct"]["Non-Black"], 2))
f.record("act2.masking.reverse_black_n", int(directions["reverse_n"]["Black"]))
f.record("act2.masking.masked_nonblack_n", int(directions["masked_n"]["Non-Black"]))
f.record("act2.masking.reverse_black_pct", round(directions["reverse_pct"]["Black"], 2))

_rev = tier_transition_table(dm, "reverse")
f.record(
    "act2.masking.reverse_out_of_contraindication",
    int(_rev.loc[_rev["from"] == TIER_LABELS[0], "n"].sum()),
)
f.record(
    "act2.masking.reverse_into_standard",
    int(_rev.loc[_rev["to"] == TIER_LABELS[2], "n"].sum()),
)

f.record("act2.decision_gate.verdict", dec_verdict)
f.record("act2.decision_gate.reason", dec_why[0] if dec_why else "")

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")